# FuelNet — Exploratory Data Analysis (EDA)
Dataset: `data/Fuel_Consumption_2000-2022.csv` (Canadian NRCan 2000-2022). This notebook prints exact columns, renames to snake_case, and inspects data quality.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pathlib, re
plt.style.use('dark_background')
sns.set_theme(style='whitegrid')
csv_path = pathlib.Path('../data/Fuel_Consumption_2000-2022.csv')
if not csv_path.exists():
    csv_path = pathlib.Path('data/Fuel_Consumption_2000-2022.csv')
if not csv_path.exists():
    csv_path = pathlib.Path(r'E:\fuelnet\data\Fuel_Consumption_2000-2022.csv')
print('CSV path:', csv_path.resolve())
print('Exists:', csv_path.exists())

In [ ]:
# 0 — Exact column names before cleaning
df_raw = pd.read_csv(csv_path, nrows=0)
print('Exact columns:', df_raw.columns.tolist())
for c in df_raw.columns:
    print(repr(c))
df = pd.read_csv(csv_path)
print('\nShape raw:', df.shape)

In [ ]:
# 0b — Rename to clean snake_case (use renamed columns everywhere from now on)
rename_map = {
    'YEAR': 'year',
    'MAKE': 'make',
    'MODEL': 'model',
    'VEHICLE CLASS': 'vehicle_class',
    'ENGINE SIZE': 'engine_size_l',
    'CYLINDERS': 'cylinders',
    'TRANSMISSION': 'transmission',
    'FUEL': 'fuel_type',
    'FUEL CONSUMPTION': 'fuel_city_l_100km',  # CITY - 9.2 L/100km in sample row
    'HWY (L/100 km)': 'fuel_hwy_l_100km',
    'COMB (L/100 km)': 'fuel_comb_l_100km',
    'COMB (mpg)': 'fuel_comb_mpg',
    'EMISSIONS': 'co2_g_km',
}
df.rename(columns=rename_map, inplace=True)
print('Renamed columns:', df.columns.tolist())
# verify all renamed
expected = ['year','make','model','vehicle_class','engine_size_l','cylinders','transmission','fuel_type','fuel_city_l_100km','fuel_hwy_l_100km','fuel_comb_l_100km','fuel_comb_mpg','co2_g_km']
assert list(df.columns) == expected, 'Rename mismatch!'
print('Rename OK — 13 columns mapped to snake_case')

In [ ]:
# 1 — first 10 rows, shape, column info
print('Shape:', df.shape)
display(df.head(10))
print(df.info())
print(df.describe().round(2).to_string())

In [ ]:
# 2 — unique values for categoricals
for col in ['make','model','vehicle_class','transmission','fuel_type']:
    uniq = sorted(df[col].dropna().unique().tolist())
    print(f"{col}: {len(uniq)} uniques")
    print(uniq[:50])
    if len(uniq) > 50:
        print(' ... truncated, showing first 50')
    print()
print('Full vehicle_class uniques:')
print(sorted(df['vehicle_class'].unique().tolist()))
print('\nFull transmission uniques:')
print(sorted(df['transmission'].unique().tolist()))
print('\nFull fuel_type uniques:')
print(sorted(df['fuel_type'].unique().tolist()))

In [ ]:
# 3 — missing values per column
print(df.isnull().sum().to_string())
print('Total missing:', int(df.isnull().sum().sum()))
# also check empty strings
for col in df.select_dtypes(include='object').columns:
    empty = (df[col].astype(str).str.strip() == '').sum()
    print(f"empty strings in {col}: {empty}")

In [ ]:
# 4 — Distributions of engine_size_l, cylinders, fuel_comb, co2
fig, axes = plt.subplots(2,2, figsize=(14,10))
fig.suptitle('Distributions (n=22556)', fontsize=14)
sns.histplot(df['engine_size_l'], bins=30, kde=True, ax=axes[0,0], color='skyblue')
axes[0,0].set_title('engine_size_l')
sns.countplot(x='cylinders', data=df, ax=axes[0,1], palette='viridis', order=sorted(df['cylinders'].unique()))
axes[0,1].set_title('cylinders')
sns.histplot(df['fuel_comb_l_100km'], bins=30, kde=True, ax=axes[1,0], color='orange')
axes[1,0].set_title('fuel_comb_l_100km')
sns.histplot(df['co2_g_km'], bins=30, kde=True, ax=axes[1,1], color='salmon')
axes[1,1].set_title('co2_g_km')
plt.tight_layout()
plt.show()
for col in ['engine_size_l','cylinders','fuel_comb_l_100km','co2_g_km']:
    print(df[col].describe().to_string())
    print()

In [ ]:
# 5 — Correlation heatmap between numeric features
numeric_cols = ['year','engine_size_l','cylinders','fuel_city_l_100km','fuel_hwy_l_100km','fuel_comb_l_100km','fuel_comb_mpg','co2_g_km']
corr = df[numeric_cols].corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, square=True)
plt.title('Correlation heatmap — numeric features')
plt.show()
print(corr.round(2).to_string())

In [ ]:
# 6 — Average co2_g_km per year (2000-2022) line chart
avg_co2 = df.groupby('year')['co2_g_km'].mean().reset_index()
plt.figure(figsize=(12,5))
plt.plot(avg_co2['year'], avg_co2['co2_g_km'], marker='o', color='cyan')
plt.fill_between(avg_co2['year'], avg_co2['co2_g_km'], alpha=0.2, color='cyan')
plt.title('Average CO2 (g/km) per year (2000-2022)')
plt.xlabel('year')
plt.ylabel('avg co2_g_km')
plt.grid(True, alpha=0.3)
plt.xticks(range(2000,2023,2))
plt.show()
print(avg_co2.to_string(index=False))
print()
print('Trend: min', avg_co2['co2_g_km'].min(), 'at year', int(avg_co2.loc[avg_co2['co2_g_km'].idxmin(),'year']))
print('Max', avg_co2['co2_g_km'].max(), 'at year', int(avg_co2.loc[avg_co2['co2_g_km'].idxmax(),'year']))

In [ ]:
# 7 — Unique make+model combinations + inconsistent naming examples
df['make_model'] = df['make'] + ' ' + df['model']
n_combos = df['make_model'].nunique()
print(f"Unique make+model combinations: {n_combos} (from {len(df)} rows)")
print(f"Unique makes: {df['make'].nunique()}, unique models: {df['model'].nunique()}")
print('\nExamples of inconsistent naming — vehicle_class casing:')
print(sorted(df['vehicle_class'].unique().tolist()))
print('\nExamples where same semantic class has different spellings:')
for pat in ['COMPACT','PICKUP','SUV','VAN','STATION','SPECIAL']:
    vals = [v for v in df['vehicle_class'].unique() if pat.lower() in v.lower()]
    print(pat, '->', vals)
print('\nModel naming inconsistencies — F-150 family:')
f150_like = sorted([m for m in df['model'].unique() if 'F-150' in m or 'F150' in m])
print(f150_like[:30])
print('Total F-like models:', len(f150_like))
print('\nAnother example — "TACOMA" vs "Tacoma" casing is not present but check lower/upper:')
print(df['model'].value_counts().head(20).to_string())
print('\nCheck if make/model have leading/trailing whitespace or hyphen variations:')
# show models with hyphens vs without for same base
print(df[df['model'].str.contains('150', na=False)]['model'].value_counts().head(20).to_string())
print('\nDuplicate combos across years:')
combo_counts = df['make_model'].value_counts()
print('Top 20 most frequent make_model:')
print(combo_counts.head(20).to_string())

## Summary placeholder — fill after running
Rows, missing, quality issues, CO2 trend will be printed here.